# Spark Structured Streaming — JSON to Delta (Bronze)

Real-time micro-batch ingestion of customer records from a landing folder into a Delta **bronze** table.

**Concepts:** `readStream` / `writeStream`, schema inference, micro-batch triggers, and checkpointing for fault tolerance.

## 1. Infer a schema from a sample record
Structured Streaming needs an explicit schema. Here we derive one from a representative JSON record with `schema_of_json` (in production this might come from a schema registry or the first Kafka message).

In [0]:

from pyspark.sql.functions import schema_of_json, from_json, col

# Sample JSON (can come from a file or Kafka message)
sample_json = """
{"customer_id": 9179, "customer_name": "Richard Cox", "date_of_birth": "1996-10-25", "telephone": "+1 6680703335", "email": "devon84@mail.com", "member_since": "2024-09-26", "created_timestamp": "2024-10-17 16:12:27"}
"""

# Infer schema from the sample
json_schema = spark.range(1).select(schema_of_json(sample_json).alias("schema")).collect()[0]["schema"]
print(json_schema)


## 2. Read the stream
`spark.readStream` tails the landing folder — every new file that lands becomes a micro-batch.

In [0]:

customers_df = (
                    spark.readStream
                         .format("json")
                         .schema(json_schema)
                         .load("/Volumes/gizmobox/landing/operational_data/customers_stream/")
)

## 3. Add lineage columns
Capture the source `file_path` and an `ingestion_date` so every row is traceable back to its file.

In [0]:
from pyspark.sql.functions import current_timestamp, col

customers_transformed_df = (
                                customers_df.withColumn("file_path", col("_metadata.file_path"))
                                            .withColumn("ingestion_date", current_timestamp())
)

## 4. Write to the Bronze Delta table
Append each micro-batch to Delta every 10 seconds. The `checkpointLocation` makes the stream restartable with exactly-once guarantees.

In [0]:
streaming_query = (
    customers_transformed_df.writeStream
        .format("delta")
        .outputMode("append")
        .trigger(processingTime='10 seconds')
        .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customers_stream/_checkpoint_stream")
        .toTable("gizmobox.bronze.customers_stream")
)

## 5. Stop the stream

In [0]:
streaming_query.stop()

## 6. Verify the results

In [0]:
%sql
select * from gizmobox.bronze.customers_stream
